# Session 17 lab: entropy, cross-entropy, temperature, top-p and perplexity

KLAi | Mathematics for AI and Machine Learning

Today you measure surprise in bits, compute entropy, cross-entropy and Kullback-Leibler (KL) divergence, reshape a softmax with temperature and cut it with top-p, check by numbers that the gradient of softmax with cross-entropy is prediction minus target, and score a toy language model with perplexity.

**How to use this notebook.** Run the cells in order with Shift and Enter. Each step has the same rhythm: **PREDICT**, **RUN**, **READ**, then **CHANGE one thing**. Type your answers in the lines that start with `>`. If a number differs from the slide called "Check your numbers", stop and find the step that went wrong.

**Words and units.** *Surprise* (self-information): -log2 of the probability of an event, in bits. *Entropy*: the average surprise. *Cross-entropy*: the average surprise when reality follows P but the model believes Q. *KL divergence*: cross-entropy minus entropy. Steps 1 to 3 use log2, so they are in bits. Steps 4 to 7 use ln, like every library, so they are in nats (1 bit = 0.693 nats). *Softmax* (Session 12): turns scores (logits) into probabilities. *Temperature* and *top-p*: settings that reshape and trim those probabilities before a word is drawn. *Perplexity*: e to the power of the mean negative log-probability.

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(suppress=True, precision=6)

---
## Step 1: Surprise in bits

The surprise of an event with probability p is -log2(p) bits. Rare events are more surprising.

In [ ]:
events = np.array([1/2, 1/8, 1/1024])      # probabilities of three events
print(events)

**PREDICT first.** How many bits of surprise do you expect for each event? Hint: 1/8 is one half three times over.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# surprise in bits of each probability (see the slide "Surprise is the negative logarithm of the probability"). numpy has np.log2
surprise_bits = None  # your code here

In [ ]:
if surprise_bits is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("surprise of 1/2, 1/8 and 1/1024 in bits", surprise_bits, [1, 3, 10], 0.0001)
    print(surprise_bits)

**READ the output.** Each time the probability is cut in half, how many bits are added to the surprise?

> My answer: 

---
## Step 2: Entropy: the average surprise

The entropy of a distribution P is the sum of p times its surprise. It is the smallest possible average cost, in bits, of describing outcomes drawn from P.

In [ ]:
P = np.array([0.5, 0.25, 0.125, 0.125])     # true probabilities of four next words
fair = np.array([0.5, 0.5])
biased = np.array([0.9, 0.1])

**PREDICT first.** Which has the larger entropy: the fair coin or the coin that lands heads 90 percent of the time?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# entropy: the average surprise of the distribution (see the entropy slides). Write it once as a function so you can reuse it
def entropy(p):
    return None  # your code here

In [ ]:
H_P, H_fair, H_biased = entropy(P), entropy(fair), entropy(biased)
if H_P is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("entropy of the four-word distribution P (bits)", H_P, 1.75, 0.0001)
    check("entropy of the fair coin (bits)", H_fair, 1.0, 0.0001)
    check("entropy of the 0.9 / 0.1 coin (bits)", H_biased, 0.469, 0.001)
    print("entropy of P:", H_P, " fair coin:", H_fair, " biased coin:", round(float(H_biased), 4))

**CHANGE one thing.** Change `biased` to np.array([0.99, 0.01]) and recompute. Does the entropy go up or down? Why does a nearly certain outcome carry little surprise? Put 0.9 and 0.1 back afterwards.

> What I saw: 

---
## Step 3: Cross-entropy and KL divergence

Reality follows P. Model A believes all four words are equally likely. Model B believes Q_B. Both are in bits. The formulas are on the slides.

In [ ]:
Q_A = np.array([0.25, 0.25, 0.25, 0.25])     # model A: uniform
Q_B = np.array([0.4, 0.3, 0.2, 0.1])         # model B

**PREDICT first.** Model B is closer to P than model A is. Will its cross-entropy be larger or smaller than model A's?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# cross_entropy(P, Q) and kl(P, Q), both in bits (see the cross-entropy and KL slides)
def cross_entropy(P, Q):
    return None  # your code here

def kl(P, Q):
    return None  # your code here

In [ ]:
ce_A, ce_B = cross_entropy(P, Q_A), cross_entropy(P, Q_B)
kl_A, kl_B = kl(P, Q_A), kl(P, Q_B)
if ce_A is None or kl_A is None or H_P is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("cross-entropy of model A", ce_A, 2.0, 0.0001)
    check("cross-entropy of model B", ce_B, 1.8007, 0.0001)
    check("KL divergence of model A", kl_A, 0.25, 0.0001)
    check("KL divergence of model B", kl_B, 0.0507, 0.0001)
    check("cross-entropy = entropy + KL (model B)", ce_B, H_P + kl_B, 0.000001)
    print("cross-entropy A:", round(float(ce_A), 4), " B:", round(float(ce_B), 4), "bits")
    print("KL A:", round(float(kl_A), 4), " B:", round(float(kl_B), 4), "bits")

**READ the output.** Cross-entropy is never below the entropy. By how much is model A above it? That gap is the KL divergence.

> My answer: 

---
## Step 4: Softmax (Session 12): scores to probabilities

A model produces one score (logit) per class: cat, dog and fox. The softmax of Session 12 turns them into probabilities that are positive and add to 1: p_i = exp(z_i) / sum(exp(z_j)). The stable form subtracts the largest logit first.

In [ ]:
classes = ["cat", "dog", "fox"]
z = np.array([2.0, 1.0, 0.1])               # logits of a 3-class model
z_big = np.array([1000.0, 1001.0, 1002.0])   # large logits

**PREDICT first.** Which class gets the largest probability? Will the probabilities add to 1?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# A softmax that does not overflow for logits like 1000 (the stable form from Session 12)
def softmax(z):
    return None  # your code here

In [ ]:
probs = softmax(z)
if probs is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("softmax of [2, 1, 0.1]", probs, [0.6590, 0.2424, 0.0986], 0.0005)
    check("softmax of the large logits", softmax(z_big), [0.0900, 0.2447, 0.6652], 0.0005)
    print("softmax of z:", np.round(probs, 4), " sum:", round(float(probs.sum()), 6))
    print("softmax of the large logits:", np.round(softmax(z_big), 4))
    with np.errstate(all="ignore"):
        print("the naive way gives:", np.exp(z_big) / np.exp(z_big).sum())

**READ the output.** The naive way prints nan. Why does subtracting the largest logit fix it without changing the answer?

> My answer: 

---
## Step 5: Temperature and top-p (nucleus) sampling

Temperature T divides every logit before the softmax. Top-p keeps the most likely words until their running total first reaches p, then rescales the kept words so they add to 1. Five next words with illustrative logits, already sorted from most to least likely.

In [ ]:
words5 = ["tea", "coffee", "water", "milk", "soup"]
logits5 = np.array([3.0, 2.6, 1.9, 1.2, 0.4])   # illustrative, sorted from largest to smallest
top_p = 0.8

**PREDICT first.** At T = 0.5, will the top class (cat) get more or less than 0.659? And how many of the five drinks will top-p = 0.8 keep?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# p_T: the softmax of z at T = 0.5, 1 and 2 (one row each). p5: the softmax of logits5 at T = 1. Then keep the most likely words until their running total first reaches top_p (np.cumsum gives running totals), and rescale the kept ones so they add to 1
p_T = None  # your code here
p5 = None  # your code here
p_nucleus = None  # your code here: the rescaled probabilities of the kept words

In [ ]:
if p_T is None or p5 is None or p_nucleus is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("top class (cat) at T = 0.5, 1 and 2", p_T[:, 0], [0.8638, 0.6590, 0.5017], 0.0005)
    check("number of drinks top-p 0.8 keeps", len(p_nucleus), 3, 0)
    check("rescaled probabilities of the kept drinks", p_nucleus, [0.4992, 0.3346, 0.1662], 0.0005)
    print("cat, dog, fox at T = 0.5, 1, 2:\n", np.round(p_T, 4))
    print("drinks at T = 1:", np.round(p5, 4), " running total:", np.round(np.cumsum(p5), 4))
    print("kept and rescaled:", dict(zip(words5, np.round(p_nucleus, 4))))

**READ the output.** A low temperature sharpens the probabilities and a high one flattens them. After the top-p cut, can milk or soup ever be drawn?

> My answer: 

**CHANGE one thing.** Apply top-p to the drinks at T = 2 instead: `softmax(logits5 / 2)`. How many drinks does top-p = 0.8 keep now, and why?

> What I saw: 

---
## Step 6: The gradient of softmax with cross-entropy is prediction minus target

Dog is the true word, so the one-hot target is y = (0, 1, 0). The loss is L = -sum(y * ln p), in nats. Work out the gradient of L with respect to the logits (the derivation slide shows it) and check it against a numeric gradient.

In [ ]:
y = np.array([0.0, 1.0, 0.0])               # dog is the true word (one-hot target)

def loss(z):
    return -np.sum(y * np.log(softmax(z)))

**PREDICT first.** Dog is true, but it only got probability 0.24. Will the gradient for the dog logit be positive or negative? (Should that logit go up or down?)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# the gradient of the loss with respect to the logits, from the derivation slide
grad = None  # your code here

In [ ]:
if grad is None or probs is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    L0 = loss(z)
    numeric = np.zeros(3)
    for j in range(3):                        # nudge one logit at a time and watch the loss
        zz = z.copy(); zz[j] += 1e-6
        numeric[j] = (loss(zz) - L0) / 1e-6
    lse_form = np.log(np.sum(np.exp(z))) - z[1]   # log-softmax: log-sum-exp (Session 12) minus the true logit
    zs = np.array([0.5, 0.0])                 # two classes: the softmax of (z1, z2) is the sigmoid of z1 - z2
    check("loss (cross-entropy, nats)", L0, 1.4170, 0.001)
    check("gradient p - y", grad, [0.6590, -0.7576, 0.0986], 0.0005)
    check("formula matches the numeric gradient (largest difference)", np.max(np.abs(grad - numeric)), 0.0, 0.00001)
    check("log-sum-exp form gives the same loss", lse_form, L0, 0.000001)
    check("two-class softmax equals the sigmoid of z1 - z2", softmax(zs)[0], 1 / (1 + np.exp(-(zs[0] - zs[1]))), 0.000001)
    print("loss:", round(float(L0), 4), "nats   log-sum-exp form:", round(float(lse_form), 4))
    print("formula p - y:", np.round(grad, 4))
    print("numeric gradient:", np.round(numeric, 4))

**READ the output.** The three gradient entries add up to 0. Why must they, when p and y each add to 1?

> My answer: 

---
## Step 7: Perplexity of a toy language model

A toy model gave these probabilities to the six real words of a sentence (illustrative, made-up numbers). The perplexity formula is on the slides.

In [ ]:
words = ["The", "cat", "sat", "on", "the", "mat"]
token_probs = np.array([0.2, 0.1, 0.25, 0.5, 0.4, 0.05])   # probability the model gave to each real word

**PREDICT first.** The probabilities are between 0.05 and 0.5. Guess the perplexity: between 1 and 20?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# the mean negative natural log of the six probabilities, then the perplexity (see the perplexity slides)
mean_nll = None  # your code here
perplexity = None  # your code here

In [ ]:
total_nll = float(np.sum(-np.log(token_probs)))
if mean_nll is None or perplexity is None:
    print("NOT YET: write your code in the cell above, then run both cells again.")
else:
    check("sum of -ln p over the six words", total_nll, 9.9035, 0.0005)
    check("mean negative log-probability (nats)", mean_nll, 1.6506, 0.0005)
    check("perplexity", perplexity, 5.2100, 0.0005)
    print("sum of -ln p:", round(total_nll, 3), " mean:", round(float(mean_nll), 4), "nats  perplexity:", round(float(perplexity), 3))
    print("in bits:", round(float(np.mean(-np.log2(token_probs))), 4), " 2 ** bits:", round(float(2 ** np.mean(-np.log2(token_probs))), 3))

**READ the output.** Perplexity is the same whether we use bits or nats. It reads as the effective number of equally likely words the model chooses between. How many?

> My answer: 

**CHANGE one thing.** Replace token_probs with a model that gives every word probability 1/50 (`np.full(6, 1/50)`). What is the perplexity? Then try all 1.0.

> What I saw: 

---
## Save your work

Run every cell again from the top (Runtime, Restart and run all), check the last line says all checks pass, then save the notebook.

In [ ]:
show_checks(22)